# Laboratory 10 — Thermodynamic potentials

In this laboratory you take module 09's fundamental relation and read it through four windows —
$U$, $H$, $F$ and $G$ — each matched to what a laboratory can actually hold fixed. Then you
measure an entropy change with nothing but a pressure gauge and a thermometer, and read the
entropy of a stretched rubber band off a force gauge.

The route: one state and every potential built from it (Part 1); the Legendre transform, done
numerically (Part 2); Maxwell relations as a cross-derivative gap, and a pair of fields that
fails the test (Part 3); a piston against a heat bath, where the free energy falls while the
energy rises (Part 4); what enthalpy keeps the books on, and what a throttle conserves (Part 5);
entropy from a pressure gauge (Part 6) and how far that measurement can be trusted (Part 7);
the rubber band (Part 8); the automated checks (Part 9); and free exploration (Part 10).

Work through it in order. Where the notebook asks you to predict, write your prediction in the
cell provided **before** running the next cell. A prediction you have committed to is the only
reliable way to discover that you were wrong.

## Model specification

| | |
|---|---|
| **System** | a substance given by a smooth fundamental relation $S(U, V, N)$ — the Sackur–Tetrode ideal gas, the Einstein solid, or the van der Waals gas — alone, against a heat bath at $T$, or pushed through a porous plug |
| **Dynamics** | none for the potentials, which are functions of state; a released piston is stepped through constrained equilibria while the bath absorbs whatever heat keeps $T$ fixed |
| **Boundary** | a heat-conducting wall to the bath; a frictionless piston where a pressure is held; an insulated porous plug for throttling |
| **Ensemble** | not applicable — the bath is a thermodynamic idealisation, and its statistical version is module 11's |
| **Ignored** | fluctuations, the finite size of any real bath, how fast anything relaxes, and every kind of work other than $P\,\mathrm{d}V$ unless stated |
| **Valid when** | the bath is much larger than the system, and $S$ is strictly concave where it is used — for the van der Waals gas, above its critical temperature |
| **Failure modes** | a relation with a dent, where one slope names several states and the Legendre transform folds (module 14); small baths (module 11); processes too fast to pass through equilibrium states |

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import fundamental, gases, potentials, processes
from thermolab.constants import AMU, K_B, N_A
from thermolab.validation import convergence_study, relative_error, seed_study

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

ARGON_MASS = 39.948 * AMU
# Argon's van der Waals constants, per particle: a_molar / N_A^2 and b_molar / N_A.
VDW_A, VDW_B = 0.1355 / N_A**2, 3.201e-5 / N_A

argon = fundamental.monatomic_ideal_gas(ARGON_MASS)
argon_vdw = potentials.van_der_waals_gas(ARGON_MASS, VDW_A, VDW_B)
print(f"k_B = {K_B:.6e} J/K")
print(f"van der Waals argon: critical temperature "
      f"{gases.vdw_critical_point(VDW_A, VDW_B)[1]:.1f} K")

## Part 1 — One state, every potential

Take one mole of argon at $300\ \mathrm{K}$ and $1\ \mathrm{bar}$. The relation supplies its
energy, entropy and pressure; every potential is then $U$ with some of $TS$ and $PV$ added or
taken away. Below the numbers, the table the module page builds: each potential, the variables
it is naturally a function of, and when it is the one to reach for.

In [ ]:
n_atoms = N_A
volume = n_atoms * K_B * 300.0 / 1e5
state = potentials.ledger_at(argon, 300.0, volume, n_atoms)

print(f"U                = {state.energy:10.1f} J")
print(f"TS               = {state.ts:10.1f} J")
print(f"PV               = {state.pv:10.1f} J")
print(f"H = U + PV       = {state.enthalpy:10.1f} J")
print(f"F = U - TS       = {state.helmholtz:10.1f} J")
print(f"G = U - TS + PV  = {state.gibbs:10.1f} J")
mu = fundamental.chemical_potential_of(argon, state.energy, volume, n_atoms)
print(f"\nG / N = {state.gibbs / n_atoms:.6e} J;  module 09's slope gives mu = {mu:.6e} J")
print()
for symbol in ("S", "U", "H", "F", "G"):
    nv = potentials.natural_variables(symbol)
    print(f"{nv.symbol}({', '.join(nv.variables)})   {nv.differential:<38}  {nv.behaviour}")

fig, ax = plt.subplots(figsize=(7, 3.2))
names = ["U", "TS", "PV", "H", "F", "G"]
values = [state.energy, state.ts, state.pv, state.enthalpy, state.helmholtz, state.gibbs]
ax.bar(names, np.array(values) / 1e3,
       color=["#111827", "#94a3b8", "#94a3b8", "#d97706", "#2563eb", "#dc2626"])
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("kJ")
ax.set_title("one mole of argon at 300 K and 1 bar")
plt.tight_layout()
plt.show()

Two things to notice. The free energies are large and negative, because $TS$ is several times
$U$ for a gas: most of what $TS$ books is not energy the gas *has*, and nothing here is a
container of anything. And $G/N$ equals the chemical potential module 09 read off the
relation's slope — the Euler relation, $U = TS - PV + \mu N$, rearranged, says $G = \mu N$.

### Predict

Commit to an answer for each of the module page's four predictions before going further — they
are what this laboratory tests:

1. A gas in a rigid box, in contact with a heat bath, has an internal constraint released. Must
   its energy go down?
2. Two cylinders of argon at room temperature hold exactly the same internal energy. Could one
   of them do more work than the other?
3. Stretch a rubber band quickly and touch it to your lip: warmer, or cooler? Then hang a
   weight from it and warm it with a hair dryer: does it stretch or contract?
4. Is $\Delta H$ equal to the heat $Q$ for every process, or only under conditions — and if so,
   which?

**Your predictions:**

1.
2.
3.
4.

## Part 2 — The Legendre transform, numerically

$U(S)$ at fixed $V$ and $N$ is a convex curve. At each point, its tangent has slope $T$ and
meets the $S = 0$ axis at $U - TS = F$. So sampling the curve, taking slopes and subtracting
gives $F$ as a function of $T$ — with no formula for $F$ supplied. The closed form it should
reproduce, $F = -N\kB T\,[\ln(V/N\lambda^3) + 1]$ with $\lambda$ the thermal wavelength, is
used only to check the answer.

In [ ]:
s_mid = state.entropy
s_grid = np.linspace(0.8 * s_mid, 1.2 * s_mid, 10_000)
u_grid = potentials.energy_at_entropy(argon, s_grid, volume, n_atoms)
t_grid, f_grid = potentials.legendre_transform(u_grid, s_grid)


def closed_form_f(t):
    wavelength = fundamental.PLANCK_H / np.sqrt(2 * np.pi * ARGON_MASS * K_B * t)
    return -n_atoms * K_B * t * (np.log(volume / (n_atoms * wavelength**3)) + 1)


fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
left.plot(s_grid, u_grid / 1e3, color="black")
left.set_xlabel("S (J/K)")
left.set_ylabel("U (kJ)")
left.set_title("U(S) at fixed V and N: convex")
right.plot(t_grid, f_grid / 1e3, color="#2563eb", lw=3, label="tangent intercepts")
right.plot(t_grid, closed_form_f(t_grid) / 1e3, color="black", ls="--", label="closed form")
right.set_xlabel("T (K)")
right.set_ylabel("F (kJ)")
right.legend()
plt.tight_layout()
plt.show()

legendre_error = np.max(np.abs(f_grid - closed_form_f(t_grid)) / np.abs(closed_form_f(t_grid)))
print(f"temperatures covered: {t_grid[0]:.0f} K to {t_grid[-1]:.0f} K")
print(f"largest relative error of F on the curve: {legendre_error:.2e}")

### How good is a coarse grid?

Each tangent's slope comes from a central difference, so each sample's $T$ is wrong at second
order in the grid spacing. Measure two different errors as the grid refines: how far each point
$(T, F)$ is from the true curve *at the temperature it should have had*, and how far it is from
the true curve *at the temperature it reports*.

In [ ]:
def legendre_errors(n_points):
    s = np.linspace(0.9 * s_mid, 1.1 * s_mid, n_points)
    u = np.asarray(potentials.energy_at_entropy(argon, s, volume, n_atoms))
    slope, intercept = potentials.legendre_transform(u, s)
    exact_t = 2 * u / (3 * n_atoms * K_B)  # the temperature this sample really has
    point = np.max(np.abs(intercept - closed_form_f(exact_t)) / np.abs(closed_form_f(exact_t)))
    curve = np.max(np.abs(intercept - closed_form_f(slope)) / np.abs(closed_form_f(slope)))
    return point, curve


grids = [25, 50, 100, 200, 400]
point_study = convergence_study(lambda k: legendre_errors(k)[0], grids, 0.0)
curve_study = convergence_study(lambda k: legendre_errors(k)[1], grids, 0.0)
for k, p_err, c_err in zip(grids, point_study.errors, curve_study.errors, strict=True):
    print(f"{k:4d} points:  off at its own T {p_err:.2e}    off the curve {c_err:.2e}")
print(f"\nobserved order: each point {point_study.observed_order:.2f},  "
      f"the curve {curve_study.observed_order:.2f}")

Each point is misplaced at second order, as a central difference should be. But the points slide
*along* the true curve rather than off it, and the curve they trace is right at fourth order.
The reason is the one that makes the transform work at all: the intercept $f - px$ is
stationary in $x$ at the true tangent point, so an error in the slope moves the point along
$F(T)$ and only the error's square moves it off.

## Part 3 — Maxwell relations as a cross-derivative gap

$\mathrm{d}F = -S\,\mathrm{d}T - P\,\mathrm{d}V$ is exact, so module 05's test says
$(\partial S/\partial V)_T = (\partial P/\partial T)_V$. On the van der Waals relation, $S$ is a
*value* of the relation at the located state and $P$ is a *ratio of its slopes*: two different
operations, which the Maxwell relation says must agree. `maxwell_check` returns the gap relative
to the size of the two cross-derivatives, at every point of a grid.

In [ ]:
t_axis = np.linspace(200.0, 600.0, 60)
v_axis = np.geomspace(1.5e-4, 5e-3, 60)
tt, vv = np.meshgrid(t_axis, v_axis)
minus_s, minus_p = potentials.helmholtz_form(argon_vdw, N_A)

steps = [4e-2, 2e-2, 1e-2, 5e-3]
gaps = [np.abs(potentials.maxwell_check(minus_s, minus_p, tt, vv, rel_step=h)) for h in steps]

fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), sharey=True)
for ax, h, gap in zip(axes, steps, gaps, strict=True):
    image = ax.pcolormesh(t_axis, v_axis * 1e3, gap, norm="log", vmin=1e-6, vmax=1e-3,
                          cmap="magma", shading="auto")
    ax.set_yscale("log")
    ax.set_title(f"h = {h:g}")
    ax.set_xlabel("T (K)")
axes[0].set_ylabel("V (L)")
fig.colorbar(image, ax=axes, label="|gap|")
plt.show()

gap_study = convergence_study(
    lambda k: float(np.max(np.abs(potentials.maxwell_check(minus_s, minus_p, tt, vv,
                                                           rel_step=1.0 / k)))),
    [25, 50, 100, 200], 0.0)
for k, err in zip(gap_study.refinements, gap_study.errors, strict=True):
    print(f"h = {1 / k:.4f}   largest |gap| = {err:.3e}")
print(f"\nobserved order = {gap_study.observed_order:.2f}   (central differences: 2)")

A check that can only pass is not a check. Pair the *ideal* gas's entropy with the *van der
Waals* pressure — two substances pretending to be one — and run the same test.

In [ ]:
minus_s_ideal, _ = potentials.helmholtz_form(argon, N_A)
nb = N_A * VDW_B
print("   V (L)     gap at h=1e-2   gap at h=1e-3    Nb / (2V - Nb)")
for v in (1.5e-4, 3e-4, 1e-3, 5e-3):
    coarse = potentials.maxwell_check(minus_s_ideal, minus_p, 300.0, v, rel_step=1e-2)
    fine = potentials.maxwell_check(minus_s_ideal, minus_p, 300.0, v, rel_step=1e-3)
    print(f"{v * 1e3:8.2f}    {float(coarse):12.6f}   {float(fine):12.6f}   "
          f"{nb / (2 * v - nb):12.6f}")

For a genuine relation the gap is only the truncation error of the differences: it falls
fourfold with every halving of the step, down to a floor of about $10^{-7}$ set by rounding.
The mismatched pair's gap does not move when the step is refined, and it sits exactly on
$Nb/(2V - Nb)$ — the ideal entropy grows as $N\kB/V$ with volume, while the van der Waals
pressure rises as $N\kB/(V - Nb)$ with temperature. The test tells you not only *that* the two
fields came from different substances, but by how much.

## Part 4 — A piston against a heat bath

### Predict

A rigid box, in contact with a bath at $300\ \mathrm{K}$, holds a piston. Side A has two moles of
argon in a quarter of the box; side B has one mole in the rest. The piston is released and
drifts, braked, until it stops. Does the argon's total energy go up, down, or stay the same?
What about its free energy?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
trace = potentials.free_energy_minimization(argon_vdw, 300.0, 4e-3, 2 * N_A, N_A,
                                            start_share=0.25, n_steps=201)
d_u = trace.energy - trace.energy[0]
d_f = trace.free_energy - trace.free_energy[0]
t_ds_system = 300.0 * trace.system_entropy_change
t_ds_bath = 300.0 * trace.bath_entropy_change
t_ds_total = 300.0 * trace.total_entropy_change

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
left.plot(trace.share, d_u, color="#dc2626", label="Delta U")
left.plot(trace.share, d_f, color="#2563eb", label="Delta F")
left.set_xlabel("V_A / V")
left.set_ylabel("J")
left.legend()
left.set_title("the argon")
right.plot(trace.share, t_ds_system, color="#111827", label="T Delta S (argon)")
right.plot(trace.share, t_ds_bath, color="#94a3b8", label="T Delta S (bath)")
right.plot(trace.share, t_ds_total, color="#d97706", lw=2.5, label="T Delta S (total)")
right.set_xlabel("V_A / V")
right.legend()
right.set_title("the entropy books, times T")
plt.tight_layout()
plt.show()

print(f"piston stops at V_A/V = {trace.share[-1]:.6f}   (equal densities: {2 / 3:.6f})")
print(f"pressures there: {trace.pressure_a[-1] / 1e5:.4f} bar | "
      f"{trace.pressure_b[-1] / 1e5:.4f} bar")
print(f"Delta U             = {d_u[-1]:+9.1f} J")
print(f"Delta F             = {d_f[-1]:+9.1f} J")
print(f"T Delta S (argon)   = {t_ds_system[-1]:+9.1f} J")
print(f"T Delta S (bath)    = {t_ds_bath[-1]:+9.1f} J")
print(f"Delta S (total)     = {trace.total_entropy_change[-1]:+9.4f} J/K")

ideal = potentials.free_energy_minimization(argon, 300.0, 4e-3, 2 * N_A, N_A,
                                            start_share=0.25, n_steps=21)
print(f"\nideal argon instead: Delta U = {ideal.energy[-1] - ideal.energy[0]:+.2e} J,  "
      f"Delta F = {ideal.free_energy[-1] - ideal.free_energy[0]:+.1f} J")

The argon's energy *rose* — by the amount its atoms' mutual attraction lost as the dense side
spread out — and the argon settled at the state of **largest** energy the piston could reach. Its
free energy fell to its smallest. Nothing is paradoxical: the bath paid for the rise, and the
bath's entropy fell by $\Delta U / T$, but the argon's entropy rose by far more. The total, argon
plus bath, climbed the whole way, and $-\Delta F / T$ is exactly that total. At fixed $T$ and $V$,
"minimise $F$" *is* "maximise the entropy of everything", said about the system alone.

The ideal gas has no attraction, so its energy at fixed temperature cannot change at all; its
free energy falls anyway. Neither gas minimised its energy.

### Same energy, different usefulness

Two cylinders each hold one mole of argon at $300\ \mathrm{K}$: one in $1\ \mathrm{L}$, one in
$10\ \mathrm{L}$. Their energies are equal. In a room at $300\ \mathrm{K}$, the most work either
can deliver is its free energy minus that of the state it would end in.

In [ ]:
small = potentials.ledger_at(argon, 300.0, 1e-3, N_A)
large = potentials.ledger_at(argon, 300.0, 10e-3, N_A)
print(f"U:  {small.energy:.1f} J  vs  {large.energy:.1f} J")
print(f"F:  {small.helmholtz:.1f} J  vs  {large.helmholtz:.1f} J")
print(f"extra work the compressed cylinder can deliver at 300 K: "
      f"{small.helmholtz - large.helmholtz:.1f} J   (R T ln 10 = "
      f"{N_A * K_B * 300.0 * np.log(10):.1f} J)")

## Part 5 — What enthalpy keeps the books on

**Heating at constant pressure.** Module 06's isobaric process computes its heat as
$C_P\,\Delta T$. Compare it with the change of $H = U + PV$ between the two end states, located
on the entropy surface.

In [ ]:
start = processes.EquilibriumState.from_temperature(10**22, 300.0, 4e-4)
heating = processes.isobaric(start, 6e-4)
h_start, h_end = (potentials.ledger_at(argon, s.temperature, s.volume, 10**22).enthalpy
                  for s in (heating.start, heating.end))
print(f"Q_P from C_P Delta T      = {heating.heat:.6f} J")
print(f"Delta H from the relation = {h_end - h_start:.6f} J")
print(f"work done ON the gas      = {heating.work_on_gas:.6f} J"
      "   (the -P Delta V the heat also pays)")

# Splitting water at 298.15 K and 1 bar, per mole, from standard tables:
delta_h, delta_g = 285.83e3, 237.13e3  # J/mol
faraday = 96485.33
print("\nelectrolysis of one mole of water, run reversibly:")
print(f"  electrical work done on the cell = Delta G = {delta_g / 1e3:.2f} kJ"
      f"   (cell voltage {delta_g / (2 * faraday):.3f} V)")
print(f"  heat drawn from the surroundings = Delta H - W_elec = {(delta_h - delta_g) / 1e3:.2f} kJ")
print(f"  Delta H                          = {delta_h / 1e3:.2f} kJ  -- not the heat")

**Throttling.** Push gas steadily through a porous plug from $50\ \mathrm{bar}$ to
$1\ \mathrm{bar}$, insulated. The enthalpy is the same on both sides; nothing else has to be.

In [ ]:
for label, relation in (("ideal argon", argon), ("van der Waals argon", argon_vdw)):
    out = potentials.throttle(relation, N_A, 300.0, 50e5, 1e5)
    print(f"{label:<20} T: {out.temperature_in:.2f} -> {out.temperature_out:.2f} K   "
          f"U: {out.energy_in:8.1f} -> {out.energy_out:8.1f} J   "
          f"H: {out.enthalpy_in:8.1f} -> {out.enthalpy_out:8.1f} J")

small_drop = potentials.throttle(argon_vdw, N_A, 300.0, 1.1e5, 1.0e5)
coefficient = small_drop.temperature_change / (1.0e5 - 1.1e5)
estimate = (2 * VDW_A / (K_B * 300.0) - VDW_B) / (2.5 * K_B)
print(f"\nJoule-Thomson coefficient at 1 bar: {coefficient * 1e5:.4f} K/bar  "
      f"(dilute van der Waals formula: {estimate * 1e5:.4f} K/bar)")

The ideal gas leaves the plug at the temperature it entered: its enthalpy depends on $T$ alone.
The van der Waals gas cools by about $18\ \mathrm{K}$ — at room temperature its atoms' attraction
wins over their size, and moving them apart costs energy that only their motion can supply.
This is how nitrogen and air are liquefied. In both cases $H$ is conserved exactly and $U$ is
not.

## Part 6 — Entropy from a pressure gauge

### Predict

You have a gauge that reads pressure to $0.2\%$ and a thermometer. You measure a gas's pressure
at nine temperatures, at each of nine volumes between $1$ and $2\ \mathrm{L}$. Can you
determine its entropy change on doubling its volume at fixed temperature — and to what
precision?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
temperatures = np.linspace(280.0, 320.0, 9)
volumes = 1e-3 * np.geomspace(1.0, 2.0, 9)
n_gas = 1e21
readings = potentials.gauge_readings(argon, n_gas, temperatures, volumes, 0.002, rng)
result = potentials.entropy_from_gauge(temperatures, volumes, readings)
exact = n_gas * K_B * np.log(2)

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
colours = plt.cm.viridis(np.linspace(0, 1, volumes.size))
for row, colour in zip(readings, colours, strict=True):
    left.plot(temperatures, row / 1e3, "o", color=colour, ms=4)
    fit = np.polyfit(temperatures, row, 1)
    left.plot(temperatures, np.polyval(fit, temperatures) / 1e3, color=colour, lw=1)
left.set_xlabel("T (K)")
left.set_ylabel("P (kPa)")
left.set_title("the gauge: one straight line per volume")
right.errorbar(volumes * 1e3, result.slopes * volumes, yerr=result.slope_errors * volumes,
               fmt="o", color="black")
right.axhline(n_gas * K_B, color="#dc2626", ls="--", label="N k_B")
right.set_xlabel("V (L)")
right.set_ylabel("V (dP/dT)_V  (J/K)")
right.legend()
right.set_title("the integrand, from the fitted slopes")
plt.tight_layout()
plt.show()

print(f"Delta S from gauge and thermometer = {result.entropy_change[-1]:.5e} "
      f"+/- {result.entropy_error[-1]:.1e} J/K")
print(f"N k_B ln 2                        = {exact:.5e} J/K")
print(f"difference, in error bars          = "
      f"{(result.entropy_change[-1] - exact) / result.entropy_error[-1]:+.2f}")
print(f"relative uncertainty               = {result.entropy_error[-1] / exact:.2%}")

The same measurement on a dense van der Waals gas: one mole, from $0.2$ to $0.4\ \mathrm{L}$.

In [ ]:
v_dense = np.geomspace(2e-4, 4e-4, 65)
readings_vdw = potentials.gauge_readings(argon_vdw, N_A, temperatures, v_dense, 0.0, rng)
dense = potentials.entropy_from_gauge(temperatures, v_dense, readings_vdw)
nb = N_A * VDW_B
print(f"Delta S from the gauge          = {dense.entropy_change[-1]:.5f} J/K")
print(f"N k_B ln((V2 - Nb)/(V1 - Nb))   = {N_A * K_B * np.log((4e-4 - nb) / (2e-4 - nb)):.5f} J/K")
print(f"N k_B ln 2 (if it were ideal)   = {N_A * K_B * np.log(2):.5f} J/K")

Neither measurement involved anything that reads entropy. The Maxwell relation turned a slope
of pressure against temperature into a slope of entropy against volume. For the dense gas the
attraction drops out entirely — at fixed volume it lowers the pressure by the same amount at
every temperature, so it does not touch $(\partial P/\partial T)_V$ — while the excluded volume
does not: the atoms have less room to be anywhere, and their entropy grows by more when the box
does.

## Part 7 — Can the error bar be trusted?

One run is an anecdote. Repeat the gauge measurement under twenty independent seeds: the mean
should sit on $N\kB \ln 2$, and the scatter between runs should match the error bar each single
run reports about itself.

In [ ]:
def gauge_delta_s(generator):
    noisy = potentials.gauge_readings(argon, n_gas, temperatures, volumes, 0.002, generator)
    return float(potentials.entropy_from_gauge(temperatures, volumes, noisy).entropy_change[-1])


study = seed_study(gauge_delta_s, n_seeds=20)
scatter = float(np.std(study.values, ddof=1))
print(f"mean over 20 seeds = {study.mean:.5e} +/- {study.standard_error:.1e} J/K   "
      f"(N k_B ln 2 = {exact:.5e})")
print(f"scatter between runs      = {scatter:.2e} J/K")
print(f"error bar a single run reports = {result.entropy_error[-1]:.2e} J/K")

## Part 8 — The rubber band

A rubber band pulled with tension $f$ has work $f\,\mathrm{d}L$ done on it as it stretches, so
its free energy obeys $\mathrm{d}F = -S\,\mathrm{d}T + f\,\mathrm{d}L$, and its Maxwell relation is

$$
\left(\frac{\partial S}{\partial L}\right)_T = -\left(\frac{\partial f}{\partial T}\right)_L .
$$

A force gauge and a thermometer measure the right-hand side. `data/10-rubber-band.csv` is a run
at fixed length (read its header for exactly what it is): a heating sweep, then a cooling sweep.

### Predict

Before running the next cell: will the tension at fixed length rise or fall as the band warms?
And what does your answer say about the sign of $(\partial S/\partial L)_T$ — does stretching a
rubber band raise its entropy or lower it?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
from pathlib import Path

try:
    import piplite  # noqa: F401
except ImportError:
    # Desktop / nbmake: the repository root is three directories up from this notebook.
    csv_path = Path("..", "..", "..", "data", "10-rubber-band.csv")
else:
    # JupyterLite bundles only the notebooks/ tree (jupyter_lite_config.json's
    # LiteBuildConfig.contents), so the browser gets its own copy of the CSV co-located here.
    csv_path = Path("data", "10-rubber-band.csv")

band = np.genfromtxt(csv_path, delimiter=",", comments="#")
band_t, band_f, sweep = band.T
warming, cooling = sweep == 1, sweep == 2


def fit(t, f):
    coeffs, cov = np.polyfit(t, f, 1, cov=True)
    return coeffs, float(np.sqrt(cov[0, 0]))


(slope, intercept), slope_error = fit(band_t, band_f)
(slope_h, _), error_h = fit(band_t[warming], band_f[warming])
(slope_c, _), error_c = fit(band_t[cooling], band_f[cooling])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(band_t[warming], band_f[warming], "o", color="#dc2626", label="heating")
ax.plot(band_t[cooling], band_f[cooling], "s", color="#2563eb", label="cooling")
line_t = np.linspace(band_t.min(), band_t.max(), 50)
ax.plot(line_t, intercept + slope * line_t, color="black", lw=1)
ax.set_xlabel("T (K)")
ax.set_ylabel("tension f (N)")
ax.legend()
plt.tight_layout()
plt.show()

t_room = 295.0
f_room = intercept + slope * t_room
print(f"(df/dT)_L, all readings  = {slope * 1e3:.2f} +/- {slope_error * 1e3:.2f} mN/K")
print(f"            heating only = {slope_h * 1e3:.2f} +/- {error_h * 1e3:.2f} mN/K")
print(f"            cooling only = {slope_c * 1e3:.2f} +/- {error_c * 1e3:.2f} mN/K")
print(f"(dS/dL)_T = -(df/dT)_L   = {-slope * 1e3:.2f} mJ/(K m)")
print(f"\nat {t_room:.0f} K the tension is {f_room:.3f} N, of which T (df/dT)_L = "
      f"{t_room * slope:.3f} N is entropic")
print(f"energetic share of the tension: {1 - t_room * slope / f_room:.1%}")

The tension rises with temperature, so stretching a rubber band at fixed temperature *lowers*
its entropy: the long molecules are pulled out of their tangled, high-multiplicity
arrangements. Everything else follows from that one sign.

- **The lip.** Stretch the band fast and no heat has time to leave, so its entropy stays put.
  The chains have fewer arrangements available, so the entropy has to be found somewhere else:
  in faster jiggling. The band warms.
- **The weight.** Hold the tension fixed and warm the band: the tension it *would* have at this
  length rises, so it pulls the weight up. A loaded band contracts on heating — the opposite of
  almost every other material.
- **The energetic share.** Only about $15\%$ of the tension is the energy of stretched bonds. The
  rest is entropy: module 08's entropic force, measured.

The heating and cooling sweeps give slopes that differ by about twice their quoted errors, and
the cooling sweep sits slightly lower: stretched rubber slowly relaxes, so the sample was not
quite the same material for the two sweeps. Fitting all readings together averages over it.

## Part 9 — Automated checks

A simulation you have not checked is a picture, not evidence. These are the same assertions
that run in the project's test suite.

In [ ]:
# 1. The ledger closes, and G = mu N (Part 1).
assert relative_error(state.energy, state.helmholtz + state.ts) < 1e-12
assert relative_error(state.gibbs / n_atoms, mu) < 1e-6

# 2. The numerical Legendre transform lies on the closed-form F(T) (Part 2) --
#    the curve at fourth order, each point at second.
assert legendre_error < 1e-6
assert 3.7 < curve_study.observed_order < 4.5
assert 1.8 < point_study.observed_order < 2.2

# 3. The Maxwell gap of a genuine relation falls at second order (Part 3) ...
assert 1.9 < gap_study.observed_order < 2.1
# ... and a mismatched pair's does not fall at all.
fake = potentials.maxwell_check(minus_s_ideal, minus_p, 300.0, 3e-4, rel_step=1e-3)
assert relative_error(float(fake), nb / (2 * 3e-4 - nb)) < 1e-3

# 4. The piston: F falls, total entropy climbs, U climbs; it stops at equal pressures (Part 4).
assert np.all(np.diff(trace.free_energy) <= 0)
assert np.all(np.diff(trace.total_entropy_change) >= 0)
assert d_u[-1] > 0
assert relative_error(trace.pressure_a[-1], trace.pressure_b[-1]) < 1e-6

# 5. Q_P = Delta H, and a throttle conserves H but cools only the real gas (Part 5).
assert relative_error(h_end - h_start, heating.heat) < 1e-8
assert relative_error(potentials.throttle(argon, N_A, 300.0, 50e5, 1e5).temperature_out,
                      300.0) < 1e-9
assert relative_error(coefficient, estimate) < 5e-3

# 6. The gauge measures N k_B ln 2, and its error bar is honest (Parts 6 and 7).
assert abs(result.entropy_change[-1] - exact) < 3.5 * result.entropy_error[-1]
assert study.agrees_with(exact, n_sigma=3.5)
assert 0.6 < scatter / result.entropy_error[-1] < 1.6

# 7. The rubber band's entropy falls as it is stretched (Part 8).
assert slope > 0 and slope > 5 * slope_error
print("all checks passed")

## Part 10 — Explore it yourself

Choose the gas, the bath temperature, how the particles are split and where the piston starts,
then press **Run Interact**. The left panel shows the gas's energy and free energy as the piston
moves; the right panel shows the entropy books. Three things worth trying:

1. Switch to the ideal gas. What happens to $\Delta U$?
2. Put the same number of particles on both sides, and start the piston far to one side.
3. Lower the temperature towards the van der Waals critical temperature, about $151\ \mathrm{K}$.
   Does $\Delta U$ grow or shrink relative to $\Delta F$?

In [ ]:
import ipywidgets as widgets


def explore(gas="van der Waals", temperature=300.0, moles_a=2.0, moles_b=1.0, start_share=0.25):
    relation = argon_vdw if gas == "van der Waals" else argon
    run = potentials.free_energy_minimization(relation, temperature, 4e-3, moles_a * N_A,
                                              moles_b * N_A, start_share, n_steps=121)
    du = run.energy - run.energy[0]
    df = run.free_energy - run.free_energy[0]
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))
    left.plot(run.share, du, color="#dc2626", label="Delta U")
    left.plot(run.share, df, color="#2563eb", label="Delta F")
    left.set_xlabel("V_A / V")
    left.set_ylabel("J")
    left.legend()
    right.plot(run.share, temperature * run.system_entropy_change, color="#111827",
               label="T Delta S (gas)")
    right.plot(run.share, temperature * run.bath_entropy_change, color="#94a3b8",
               label="T Delta S (bath)")
    right.plot(run.share, temperature * run.total_entropy_change, color="#d97706", lw=2.5,
               label="T Delta S (total)")
    right.set_xlabel("V_A / V")
    right.legend()
    plt.tight_layout()
    plt.show()
    print(f"piston stops at V_A/V = {run.share[-1]:.4f};  Delta U = {du[-1]:+.1f} J,  "
          f"Delta F = {df[-1]:+.1f} J,  Delta S_total = {run.total_entropy_change[-1]:+.4f} J/K")


widgets.interact_manual(
    explore,
    gas=["van der Waals", "ideal"],
    temperature=widgets.FloatSlider(value=300.0, min=160.0, max=600.0, step=10.0),
    moles_a=widgets.FloatSlider(value=2.0, min=0.2, max=4.0, step=0.1),
    moles_b=widgets.FloatSlider(value=1.0, min=0.2, max=4.0, step=0.1),
    start_share=widgets.FloatSlider(value=0.25, min=0.05, max=0.95, step=0.01),
);

## Check your understanding

Run the cell below for the auto-graded quiz. The same questions, with written explanations
for every option, are on the module page.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "10-potentials.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. What did you predict that turned out to be wrong, and what specifically was the flaw in
   your reasoning?
2. In Part 4 the argon ended at the *highest* energy the piston could reach. Explain, in terms
   of the bath, why that did not violate anything.
3. The gauge in Part 6 never measured an entropy. Say exactly which step turned pressure and
   temperature readings into one.
4. A student says the rubber band's tension is "just like a spring's". Use your Part 8 numbers
   to say what is right and what is wrong about that.

**Your answers:**

1.
2.
3.
4.